In [13]:
from groundingdino.util.inference import load_model, load_image, predict, annotate
import cv2

DEVICE = "cpu"

# model = load_model("groundingdino/config/GroundingDINO_SwinT_OGC.py", "weights/groundingdino_swint_ogc.pth", device=DEVICE)

model = load_model("groundingdino/config/GroundingDINO_SwinB_cfg.py", "weights/groundingdino_swinb_cogcoor.pth", device=DEVICE)

IMAGE_PATH = "assets/nid_45.png"
TEXT_PROMPT = "head . wing . fuselage . engine . tail"
BOX_TRESHOLD = 0.3
TEXT_TRESHOLD = 0.25

image_source, image = load_image(IMAGE_PATH)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD,
    device=DEVICE
)

annotated_frame = annotate(
    image_source=image_source,
    boxes=boxes,
    logits=logits,
    phrases=phrases
)

cv2.imwrite("annotated_image_swinb.jpg", annotated_frame)
print("Saved successfully")

final text_encoder_type: bert-base-uncased
Saved successfully


# Aircraft Dataset

In [ ]:
import os
import cv2
import torch
import torchvision
import json
from collections import Counter
from torchvision.ops import box_convert, batched_nms
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

# Auto-detect Hardware
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Model
model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

INPUT_DIR = "Aircraft"
OUTPUT_DIR = "Annotated_Aircraft"
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "aircraft_detection_summary.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT_GROUPS =[
    # Group 1: Main Body
    "airplane fuselage . cockpit",
    
    # Group 2: Wings
    "airplane wing",
    
    # Group 3: Propulsion
    "jet engine . airplane propeller",
    
    # Group 4: Tail Section
    "airplane tail . horizontal stabilizer . vertical stabilizer"
]

BOX_THRESHOLD = 0.28
TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.45

image_list =[
    img for img in os.listdir(INPUT_DIR)
    if img.lower().endswith((".jpg", ".png", ".jpeg"))
]

# Dictionary to hold our JSON data
detection_summary = {}

progress_bar = tqdm(image_list, desc="Processing images")

for img_name in progress_bar:
    progress_bar.set_postfix(file=img_name)
    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)

    image_source, image = load_image(IMAGE_PATH)
    
    all_boxes = []
    all_logits =[]
    all_phrases =[]

    for prompt in PROMPT_GROUPS:
        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=prompt,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            device=DEVICE
        )
        
        if len(boxes) > 0:
            all_boxes.append(boxes)
            all_logits.append(logits)
            all_phrases.extend(phrases)

    if len(all_boxes) == 0:
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        detection_summary[img_name] = {} # Record empty detection
        continue

    all_boxes = torch.cat(all_boxes, dim=0)
    all_logits = torch.cat(all_logits, dim=0)

    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy")

    unique_phrases = list(set(all_phrases))
    phrase_to_id = {phrase: i for i, phrase in enumerate(unique_phrases)}
    class_idxs = torch.tensor([phrase_to_id[p] for p in all_phrases], device=all_boxes.device)

    keep_indices = batched_nms(
        boxes=boxes_xyxy,
        scores=all_logits,
        idxs=class_idxs,
        iou_threshold=NMS_IOU_THRESHOLD
    )

    final_boxes = all_boxes[keep_indices]
    final_logits = all_logits[keep_indices]
    final_phrases = [all_phrases[i] for i in keep_indices]

    # --- NEW: Count the detected phrases and save to dictionary ---
    phrase_counts = dict(Counter(final_phrases))
    detection_summary[img_name] = phrase_counts

    annotated_frame = annotate(
        image_source=image_source,
        boxes=final_boxes,
        logits=final_logits,
        phrases=final_phrases
    )

    output_path = os.path.join(OUTPUT_DIR, img_name)
    cv2.imwrite(output_path, annotated_frame)

# --- NEW: Save the dictionary to a JSON file ---
with open(JSON_OUTPUT_FILE, "w") as json_file:
    json.dump(detection_summary, json_file, indent=4)

print(f"All images processed. Summary saved to {JSON_OUTPUT_FILE}")

# Car Dataset

In [ ]:
import os
import cv2
import torch
import torchvision
import json
from collections import Counter
from torchvision.ops import box_convert, batched_nms
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

# 1. Auto-detect Hardware
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# Model
model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

# Paths
INPUT_DIR = "car_hidream"
# Updated output folder name to reflect the optimizations
OUTPUT_DIR = "Annotated_Car_Optimized_NMS" 
os.makedirs(OUTPUT_DIR, exist_ok=True)

# JSON file will be saved neatly inside the new output folder
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "car_detection_summary.json")

# 2. Optimized and Split Prompts (Singular, No Parentheses, Grouped Logically)
PROMPT_GROUPS =[
    # Group 1: Front and Main Glass
    "car hood . car windshield . car headlight . car grille",
    
    # Group 2: Sides and Wheels
    "car wheel . car door . car fender . side window . rearview mirror",
    
    # Group 3: Rear and Roof
    "car trunk . car taillight . car roof",
    
    # Group 4: Bumpers (Separated to avoid confusion)
    "front bumper . rear bumper"
]

BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.45 # Batched NMS threshold

image_list =[
    img for img in os.listdir(INPUT_DIR)
    if img.lower().endswith((".jpg", ".png", ".jpeg"))
]

# Dictionary to hold our JSON data
detection_summary = {}

progress_bar = tqdm(image_list, desc="Processing images")

for img_name in progress_bar:
    progress_bar.set_postfix(file=img_name)
    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)

    # Load image
    image_source, image = load_image(IMAGE_PATH)
    
    all_boxes = []
    all_logits =[]
    all_phrases =[]

    # 3. Loop through prompt groups
    for prompt in PROMPT_GROUPS:
        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=prompt,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            device=DEVICE
        )
        
        if len(boxes) > 0:
            all_boxes.append(boxes)
            all_logits.append(logits)
            all_phrases.extend(phrases)

    # If nothing was detected, save original image and record empty dict
    if len(all_boxes) == 0:
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        detection_summary[img_name] = {}
        continue

    # Concatenate all predictions
    all_boxes = torch.cat(all_boxes, dim=0)
    all_logits = torch.cat(all_logits, dim=0)

    # 4. Box Conversion for NMS
    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy")

    # 5. Class-Specific NMS (Batched NMS)
    # This ensures a "car door" box doesn't delete a "side window" box inside it
    unique_phrases = list(set(all_phrases))
    phrase_to_id = {phrase: i for i, phrase in enumerate(unique_phrases)}
    class_idxs = torch.tensor([phrase_to_id[p] for p in all_phrases], device=all_boxes.device)

    keep_indices = batched_nms(
        boxes=boxes_xyxy,
        scores=all_logits,
        idxs=class_idxs,
        iou_threshold=NMS_IOU_THRESHOLD
    )

    final_boxes = all_boxes[keep_indices]
    final_logits = all_logits[keep_indices]
    final_phrases = [all_phrases[i] for i in keep_indices]

    # 6. Count the detected phrases and save to dictionary
    phrase_counts = dict(Counter(final_phrases))
    detection_summary[img_name] = phrase_counts

    # Annotate
    annotated_frame = annotate(
        image_source=image_source,
        boxes=final_boxes,
        logits=final_logits,
        phrases=final_phrases
    )

    # Save output image
    output_path = os.path.join(OUTPUT_DIR, img_name)
    cv2.imwrite(output_path, annotated_frame)

# 7. Save the dictionary to a JSON file
with open(JSON_OUTPUT_FILE, "w") as json_file:
    json.dump(detection_summary, json_file, indent=4)

print(f"All images processed successfully. Summary saved to {JSON_OUTPUT_FILE}")

# Clothing Dataset

In [ ]:
import os
import cv2
import torch
import torchvision
import json
from collections import Counter
from torchvision.ops import box_convert
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

# Auto-detect Hardware
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Model
model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

INPUT_DIR = "clothing_dataset"
OUTPUT_DIR = "Annotated_Clothing_NMS_Filtered"
JSON_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "clothing_detection_summary.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPT_GROUPS =[
    # Group 1: Upper body main structure
    "torso . neckline . shirt collar . shoulder seam",
    
    # Group 2: Arms and openings
    "left sleeve . right sleeve . sleeve cuff . armhole",
    
    # Group 3: Lower body
    "left pant leg . right pant leg . pants crotch",
    
    # Group 4: Edges and fasteners
    "waistline . hem . zipper"
]

BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.5

image_list =[
    img for img in os.listdir(INPUT_DIR)
    if img.lower().endswith((".jpg", ".png", ".jpeg"))
]

# Dictionary JSON data
detection_summary = {}

progress_bar = tqdm(image_list, desc="Processing images")

for img_name in progress_bar:
    progress_bar.set_postfix(file=img_name)
    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)
    
    image_source, image = load_image(IMAGE_PATH)
    
    all_boxes =[]
    all_logits = []
    all_phrases =[]
    
    for prompt in PROMPT_GROUPS:
        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=prompt,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            device=DEVICE
        )
        
        if len(boxes) > 0:
            all_boxes.append(boxes)
            all_logits.append(logits)
            all_phrases.extend(phrases)
            
    if len(all_boxes) == 0:
        cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), image_source)
        detection_summary[img_name] = {} # Record empty detection
        continue

    all_boxes = torch.cat(all_boxes, dim=0)
    all_logits = torch.cat(all_logits, dim=0)
    
    boxes_xyxy = box_convert(all_boxes, in_fmt="cxcywh", out_fmt="xyxy")
    keep_indices = torchvision.ops.nms(boxes_xyxy, all_logits, NMS_IOU_THRESHOLD)
    
    final_boxes = all_boxes[keep_indices]
    final_logits = all_logits[keep_indices]
    final_phrases = [all_phrases[i] for i in keep_indices]

    # save to dictionary
    phrase_counts = dict(Counter(final_phrases))
    detection_summary[img_name] = phrase_counts

    annotated_frame = annotate(
        image_source=image_source,
        boxes=final_boxes,
        logits=final_logits,
        phrases=final_phrases
    )
    
    output_path = os.path.join(OUTPUT_DIR, img_name)
    cv2.imwrite(output_path, annotated_frame)

with open(JSON_OUTPUT_FILE, "w") as json_file:
    json.dump(detection_summary, json_file, indent=4)

print(f"All images processed. Summary saved to {JSON_OUTPUT_FILE}")

final text_encoder_type: bert-base-uncased


Processing images:  14%|█▍        | 12/84 [11:32<1:10:21, 58.63s/it, file=30.jpg]   